In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import StandardScaler
import time
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
#from sklearn.naive_bayes import GaussianNB -- cannot be used in RFE
#from sklearn.neighbors import KNeighborsClassifier  -- cannot be used in RFE


In [2]:
##Independent variable - x ; Dependent variable- y; Number of featues - n/k


def rfeFeature(x,y,n):
    rfelist=[]
    LR=LogisticRegression(solver='saga',max_iter=1000, random_state=42)
    RF=RandomForestClassifier(n_estimators=10,criterion='entropy',random_state=0)
    #NB=GaussianNB()  -- cannot be used in RFE
    DT=DecisionTreeClassifier(criterion='gini',max_features='sqrt',splitter='best',random_state=0)
    SV=SVC(kernel='linear',random_state=0)
    
    for i in [LR,RF,DT,SV]:
        print(i)
        rfe=RFE(i,n)
        rfe_fit=rfe.fit(x,y)
        rfe_feature=rfe_fit.transform(x)
        rfelist.append(rfe_feature)
        print(rfelist)
    return rfelist

def split_scalar(x,y):
    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25,random_state=0)
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    return x_train,x_test,y_train,y_test

def cm_prediction(classifier,x_test):
    y_pred=classifier.predict(x_test)
    
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(y_test,y_pred)
    
    from sklearn.metrics import classification_report
    report=classification_report(y_test,y_pred)
    
    from sklearn.metrics import accuracy_score
    Accuracy=accuracy_score(y_test,y_pred)
    
    return classifier,Accuracy,report,x_test,y_test,cm

def logistic(x_train,y_train,x_test):
    from sklearn.linear_model import LogisticRegression
    classifier=LogisticRegression(random_state=0)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def svm_linear(x_train,y_train,x_test):
    from sklearn.svm import SVC
    classifier=SVC(kernel='linear',random_state=0)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def svm_NL(x_train,y_train,x_test):
    from sklearn.svm import SVC
    classifier=SVC(kernel='rbf',random_state=0)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def Navie(x_train,y_train,x_test):
    from sklearn.naive_bayes import GaussianNB
    classifier=GaussianNB()
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def knn(x_train,y_train,x_test):
    from sklearn.neighbors import KNeighborsClassifier
    classifier=KNeighborsClassifier(n_neighbors=5,metric='minkowski',p=2)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def decision(x_train,y_train,x_test):
    from sklearn.tree import DecisionTreeClassifier
    classifier=DecisionTreeClassifier(criterion='entropy',random_state=0)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def random(x_train,y_train,x_test):
    from sklearn.ensemble import RandomForestClassifier
    classifier=RandomForestClassifier(n_estimators=10 , criterion='entropy', random_state=0)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def rfe_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf):
    dataframe=pd.DataFrame(index=['LogR','SVM','RF','DT'],columns=['Logistic','SVMl','SVMnl','KNN','Naive','Decision','Random'])
    for number, index in enumerate(dataframe.index):
        #print("number, index",number, index)
        dataframe['Logistic'][index]=acclog[number]
        dataframe['SVMl'][index]=accsvml[number]
        dataframe['SVMnl'][index]=accsvmnl[number]
        dataframe['KNN'][index]=accknn[number]
        dataframe['Naive'][index]=accnav[number]
        dataframe['Decision'][index]=accdes[number]
        dataframe['Random'][index]=accrf[number]
    return dataframe


        
            

In [3]:
dataset1=pd.read_csv('watson_healthcare_modified.csv',index_col=None)
df2=dataset1


In [4]:
df2=pd.get_dummies(df2,drop_first=True)
df2

,EmployeeID,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,...,EducationField_Other,EducationField_Technical Degree,Gender_Male,JobRole_Administrative,JobRole_Nurse,JobRole_Other,JobRole_Therapist,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes
0,1313919,41,1102,1,2,1,2,94,3,2,...,0,0,0,0,1,0,0,0,1,1
1,1200302,49,279,8,1,1,3,61,2,2,...,0,0,1,0,0,1,0,1,0,0
2,1060315,37,1373,2,2,1,4,92,2,1,...,1,0,1,0,1,0,0,0,1,1
3,1272912,33,1392,3,4,1,4,56,3,1,...,0,0,0,0,0,1,0,1,0,1
4,1414939,27,591,2,1,1,1,40,3,1,...,0,0,1,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1671,1117656,26,471,24,3,1,3,66,1,1,...,0,1,1,0,1,0,0,0,1,1
1672,1152327,46,1125,10,3,1,3,94,2,3,...,0,0,0,0,1,0,0,1,0,1
1673,1812428,20,959,1,3,1,4,83,2,1,...,0,0,0,0,0,1,0,0,1,0
1674,1812429,39,466,1,1,1,4,65,2,4,...,0,0,0,0,0,0,1,1,0,0


In [5]:
x=df2.drop('Attrition_Yes',axis=1)
y=df2['Attrition_Yes']

In [ ]:
rfelist=rfeFeature(x,y,5)
rfelist

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=1000,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=42, solver='saga', tol=0.0001, verbose=0,
                   warm_start=False)


/Users/Max/opt/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/Users/Max/opt/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/Users/Max/opt/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/Users/Max/opt/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "
/Users/Max/opt/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: Conve

[array([[1313919,      41,    1102,    5993,   19479],
       [1200302,      49,     279,    5130,   24907],
       [1060315,      37,    1373,    2090,    2396],
       ...,
       [1812428,      20,     959,    2836,   11757],
       [1812429,      39,     466,   12742,    7060],
       [1152329,      27,     511,    6500,   26997]])]
RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                       criterion='entropy', max_depth=None, max_features='auto',
                       max_leaf_nodes=None, max_samples=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=10,
                       n_jobs=None, oob_score=False, random_state=0, verbose=0,
                       warm_start=False)
[array([[1313919,      41,    1102,    5993,   19479],
       [1200302,      49,     279,    5130,   24907],
      

In [ ]:
acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

In [ ]:
for i in rfelist:
    
    x_train,x_test,y_train,y_test=split_scalar(i,y)

    classifier,Accuracy,report,x_test,y_test,cm=logistic(x_train,y_train,x_test)
    acclog.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=svm_linear(x_train,y_train,x_test)
    accsvml.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=svm_NL(x_train,y_train,x_test)
    accsvmnl.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=Navie(x_train,y_train,x_test)
    accnav.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=knn(x_train,y_train,x_test)
    accknn.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=decision(x_train,y_train,x_test)
    accdes.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=random(x_train,y_train,x_test)
    accrf.append(Accuracy)
    


In [ ]:
result=rfe_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)


In [ ]:
result # n=5

In [ ]:
#result # n=4

In [ ]:
##result # n=3

In [ ]:
## On comparing with various n values for RFE, n=5 gives better result